# Parcel detector — YOLO11n training

Train the `Boxes` localization stage used before parcel condition classification. Use a Colab T4 GPU and run every cell in order.

In [ ]:
!nvidia-smi
!pip install -q ultralytics==8.4.129

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, yaml, torch

assert torch.cuda.is_available(), 'GPU is unavailable. Select Runtime > Change runtime type > T4 GPU.'
uploaded = files.upload()
expected = 'parcel_detector_dataset_v3.zip'
assert expected in uploaded, f'Upload {expected}; received: {list(uploaded)}'
dataset_root = Path('/content/parcel_detector_dataset_v3')
if dataset_root.exists(): shutil.rmtree(dataset_root)
shutil.unpack_archive('/content/' + expected, dataset_root)
print('Dataset extracted to', dataset_root)

In [ ]:
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split, expected_count in [('train', 2486), ('valid', 622)]:
    images = [p for p in (dataset_root/split/'images').iterdir() if p.suffix.lower() in image_exts]
    labels = list((dataset_root/split/'labels').glob('*.txt'))
    assert len(images) == expected_count, (split, len(images), expected_count)
    assert len(labels) == expected_count, (split, len(labels), expected_count)
    for label in labels:
        for line in label.read_text().splitlines():
            parts = line.split()
            assert len(parts) == 5 and parts[0] == '0', (label, line)
            values = [float(v) for v in parts[1:]]
            assert all(0 <= v <= 1 for v in values) and values[2] > 0 and values[3] > 0
    print(split, len(images), 'images and labels: PASS')

data_yaml = Path('/content/parcel_detector_data.yaml')
data_yaml.write_text(yaml.safe_dump({
    'path': str(dataset_root), 'train': 'train/images', 'val': 'valid/images',
    'nc': 1, 'names': ['Boxes']
}, sort_keys=False))
print(data_yaml.read_text())

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data=str(data_yaml), imgsz=320, epochs=50, batch=64, device=0, workers=2,
    pretrained=True, seed=42, deterministic=True, patience=15,
    project='/content/runs', name='PARCEL-DET-001_yolo11n_v3',
    exist_ok=False, plots=True, val=True, save=True
)

In [ ]:
run_dir = Path('/content/runs/PARCEL-DET-001_yolo11n_v3')
best = run_dir/'weights'/'best.pt'
assert best.is_file(), 'Training did not produce best.pt'
best_model = YOLO(str(best))
assert list(best_model.names.values()) == ['Boxes'], best_model.names
metrics = best_model.val(data=str(data_yaml), imgsz=320, device=0, plots=True)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
print('precision:', metrics.box.mp)
print('recall:', metrics.box.mr)

In [ ]:
checkpoint_output = Path('/content/parcel_detector_best.pt')
shutil.copy2(best, checkpoint_output)
archive = shutil.make_archive('/content/parcel_detector_training_results', 'zip', run_dir)
print(checkpoint_output, checkpoint_output.stat().st_size, 'bytes')
print(archive)
files.download(str(checkpoint_output))
files.download(archive)